In [1]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# 1. Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # Increased for better gradient stability
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 30

# 2. Enhanced Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Use actual preprocessing function for validation
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "dataset/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

# 3. Model Architecture
def build_model():
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

    # Freeze the base model for stage 1
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(3, activation="softmax")(x)

    return Model(inputs=base_model.input, outputs=output), base_model

model, base_model = build_model()

# 4. Stage 1: Training the Head Only
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Starting Stage 1: Training Top Layers...")
model.fit(train_gen, epochs=EPOCHS_STAGE1, validation_data=val_gen)

# 5. Stage 2: Full Model Fine-Tuning
print("Starting Stage 2: Fine-Tuning Entire Model...")
base_model.trainable = True # Unfreeze all layers

# Use a very small learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

# Advanced Callbacks
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

history = model.fit(
    train_gen,
    epochs=EPOCHS_STAGE2,
    validation_data=val_gen,
    callbacks=callbacks
)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/train'

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# 1. SETUP PATHS (Adjust these to your local machine)
TRAIN_PATH = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train"
VAL_PATH = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val"

# 2. ENHANCED AUGMENTATION
# Adding brightness and shear helps the model generalize better to different X-ray exposures
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=(224, 224), batch_size=32, class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    VAL_PATH, target_size=(224, 224), batch_size=32, class_mode="categorical"
)

# 3. ARCHITECTURE OPTIMIZATION
def build_model():
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

    # Stage 1: Freeze base model
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x) # Stabilizes the activations
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(3, activation="softmax")(x)

    return Model(inputs=base_model.input, outputs=output), base_model

model, base_model = build_model()

# 4. TRAINING STAGE 1: Top Layers (Warm-up)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=10, validation_data=val_gen)

# 5. TRAINING STAGE 2: Full Fine-Tuning with Label Smoothing
# Unfreeze the last 100 layers of DenseNet for specific medical feature extraction
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

# Label smoothing prevents overconfidence and helps reach higher accuracy
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
]

model.fit(train_gen, epochs=30, validation_data=val_gen, callbacks=callbacks)

Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.
Epoch 1/10
